==================================================================
Air Quality Data Normalization & Train-Test Split Pipeline - README
==================================================================

1. Libraries Imported:
   - numpy, pandas, seaborn, matplotlib.pyplot, yaml, os
   - Purpose: Data manipulation, plotting, configuration parsing, file handling

2. Load Configuration:
   - Reads 'config.yaml' to get path for processed data
   - processed_path: folder to save normalized train/test datasets

3. Load Cleaned Dataset:
   - cleaned_df_path = processed_path/cleaned_data.csv
   - clean_df = pd.read_csv(cleaned_df_path)
   - Contains numeric sensor readings, pollutant ground truth, meteorological features, and time_index

4. Select Features and Targets:
   - Features (X): sensor_co, sensor_nox, sensor_no2, temperature, relative_humidity, absolute_humidity
   - Targets (y): co_ground_truth, nox_ground_truth, no2_ground_truth

5. Train-Test Split:
   - 90% of data used for training, 10% for testing
   - Split is chronological (first 90% → train, last 10% → test) to preserve time series order

6. Compute Training Statistics:
   - X_train_mean, X_train_std, y_train_mean, y_train_std
   - Used to standardize both train and test datasets
   - Ensures no data leakage from test set

7. Standardize Features and Targets:
   - X_train_norm = (X_train - X_train_mean)/X_train_std
   - X_test_norm = (X_test - X_train_mean)/X_train_std
   - y_train_norm = (y_train - y_train_mean)/y_train_std
   - y_test_norm = (y_test - y_train_mean)/y_train_std

8. Add Normalized Time Index:
   - norm_time_index = time_index / train_time_max
   - Added as first column in X_train_norm and X_test_norm
   - Ensures model sees temporal position without dominating scales

9. Plot Histograms of Normalized Features (Optional):
   - X_train_norm.hist()
   - Visual check of feature distributions
   - Saved figure: 'Histogram of normalized X_train features'

10. Save Normalized Datasets:
    - X_train_norm.csv, X_test_norm.csv
    - y_train_norm.csv, y_test_norm.csv
    - Stored in processed_path for downstream modeling

11. Optional Visualization of Targets:
    - Histogram of y_train_norm columns
    - Saved as 'Y_train_norm_histograms.png'

12. Final Output:
    - X_train_norm / X_test_norm: normalized features including time index
    - y_train_norm / y_test_norm: normalized target pollutant concentrations
    - Ready for LSTM or Neural ODE sequence modeling

==================================================================


In [23]:
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
import os
import matplotlib.pyplot as plt 

In [24]:
# Load config.yaml
import os
from pathlib import Path
# Config paths are relative to the repository root, so run from there whatever the notebook's own folder is
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "config.yaml").exists())
os.chdir(_ROOT)
with open('configs/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [25]:
processed_path = config['data']['processed']

In [26]:
cleaned_df_path = os.path.join(processed_path, 'cleaned_data.csv')
clean_df = pd.read_csv(cleaned_df_path)
clean_df

,time_index,co_ground_truth,sensor_co,nox_ground_truth,sensor_nox,no2_ground_truth,sensor_no2,temperature,relative_humidity,absolute_humidity
0,0,2.6,1360.0,166.0,1056.0,113.0,1692.0,13.6,48.9,0.7578
1,1,2.0,1292.0,103.0,1174.0,92.0,1559.0,13.3,47.7,0.7255
2,2,2.2,1402.0,131.0,1140.0,114.0,1555.0,11.9,54.0,0.7502
3,3,2.2,1376.0,172.0,1092.0,122.0,1584.0,11.0,60.0,0.7867
4,4,1.6,1272.0,131.0,1205.0,116.0,1490.0,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...
9352,9352,3.1,1314.0,472.0,539.0,190.0,1374.0,21.9,29.3,0.7568
9353,9353,2.4,1163.0,353.0,604.0,179.0,1264.0,24.3,23.7,0.7119
9354,9354,2.4,1142.0,293.0,603.0,175.0,1241.0,26.9,18.3,0.6406
9355,9355,2.1,1003.0,235.0,702.0,156.0,1041.0,28.3,13.5,0.5139


In [27]:
features = ['sensor_co', 'sensor_nox', 'sensor_no2', 
                'temperature', 'relative_humidity', 'absolute_humidity']
X = clean_df[features]

targets = ['co_ground_truth', 'nox_ground_truth', 'no2_ground_truth']
y = clean_df[targets]

In [28]:
#Train tes split 
train_size = int(0.9 * len(clean_df))

X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

In [29]:
#Standardization
X_train_mean = X_train.mean()
X_train_std = X_train.std()
y_train_mean = y_train.mean()
y_train_std = y_train.std()

X_train_norm = (X_train - X_train_mean) / X_train_std
X_test_norm = (X_test - X_train_mean) / X_train_std
 
y_train_norm = (y_train - y_train_mean) / y_train_std
y_test_norm = (y_test - y_train_mean) / y_train_std


#Adding normalized time indices to X_train_norm and X_test_norm
train_time_norm = clean_df['time_index'][:train_size].max()
X_train_norm['norm_time_index'] = clean_df['time_index'][:train_size] / train_time_norm
X_test_norm['norm_time_index'] = clean_df['time_index'][train_size:] / train_time_norm
X_test_norm

#Shift normalized time to the first column
X_train_norm =  [col for col in X_train_norm.columns if col != 'norm_time_index']
X_test_norm =  [col for col in X_test_norm.columns if col != 'norm_time_index']


In [30]:
# Save X and y train test normalized datasets
X_train_norm.to_csv(os.path.join(processed_path, 'X_train_norm.csv'), index=False)
X_test_norm.to_csv(os.path.join(processed_path, 'X_test_norm.csv'), index=False)
y_train_norm.to_csv(os.path.join(processed_path, 'y_train_norm.csv'), index=False)
y_test_norm.to_csv(os.path.join(processed_path, 'y_test_norm.csv'), index=False)

print("All normalized datasets saved successfully!")

All normalized datasets saved successfully!


In [31]:
X_train_norm

,sensor_co,sensor_nox,sensor_no2,temperature,relative_humidity,absolute_humidity
0,1.171765,0.830898,0.619416,-0.597211,-0.011949,-0.730548
1,0.862920,1.290247,0.223478,-0.631362,-0.081614,-0.811024
2,1.362522,1.157892,0.211570,-0.790736,0.284124,-0.749484
3,1.244434,0.971039,0.297902,-0.893191,0.632446,-0.658543
4,0.772083,1.410924,0.018066,-0.870423,0.609225,-0.653311
...,...,...,...,...,...,...
8416,0.345151,-0.788501,-0.761903,-1.678678,2.019929,-0.882780
8417,-0.040905,-0.445936,-1.017923,-1.621758,1.909627,-0.875306
8418,-0.190786,-0.282439,-1.092347,-1.519304,1.764493,-0.815260
8419,0.086266,-0.566612,-0.827396,-1.382697,1.572915,-0.745248


In [32]:
"""#Visualization of normalized train-test distributions (not madatory)
# Choose dataset
df = y_train_norm  # or X_test_norm, y_train_norm, y_test_norm

num_cols = df.shape[1]

# Set figure size dynamically: width fixed, height proportional to number of columns
fig, axes = plt.subplots(num_cols, 1, figsize=(8, num_cols*2))  # 2 units height per column

# Loop through columns and plot histogram
for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='blue', alpha=0.6)
    axes[i].set_title(f'{col} distribution')
    axes[i].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(processed_path, 'Y_train_norm_histograms.png'))
plt.show()"""

"#Visualization of normalized train-test distributions (not madatory)\n# Choose dataset\ndf = y_train_norm  # or X_test_norm, y_train_norm, y_test_norm\n\nnum_cols = df.shape[1]\n\n# Set figure size dynamically: width fixed, height proportional to number of columns\nfig, axes = plt.subplots(num_cols, 1, figsize=(8, num_cols*2))  # 2 units height per column\n\n# Loop through columns and plot histogram\nfor i, col in enumerate(df.columns):\n    axes[i].hist(df[col], bins=30, color='blue', alpha=0.6)\n    axes[i].set_title(f'{col} distribution')\n    axes[i].grid(True, linestyle='--', alpha=0.3)\n\nplt.tight_layout()\nplt.savefig(os.path.join(processed_path, 'Y_train_norm_histograms.png'))\nplt.show()"